# Kabadiwala Connect: E-Waste Dataset Quality & Integrity Profiling
### Smart India Hackathon PS 26229 (Ministry of Mines / JNARDDC)

This interactive notebook provides **data quality profiling**, **distribution analysis**, and **privacy auditing** for the Kabadiwala Connect e-waste pricing dataset across target districts of Maharashtra (**Mumbai, Thane, Palghar, Pune, Nashik, Nagpur**).

The dataset pipeline enforces:
1. **Empirical & Synthetic Generation** with seasonal noise and commodity trend modeling.
2. **Multi-Stage Validation** (positivity, unit sanity, weight bounds, geo-fence, future timestamp rejection).
3. **Cleaning & Normalization** (IQR + MAD Robust Z-score outlier detection, unit conversion to INR/kg).
4. **Salted Anonymization** (HMAC-hashed collector IDs, ~500m GPS grid coarsening).

In [1]:
import json
from pathlib import Path
import numpy as np

# Locate dataset directories
notebook_dir = Path.cwd()
data_dir = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir / "data"

raw_path = data_dir / "raw" / "raw_prices_latest.json"
cleaned_path = data_dir / "processed" / "cleaned_prices_latest.json"
metrics_path = data_dir / "processed" / "quality_metrics_latest.json"
board_path = data_dir / "processed" / "rolling_board_latest.json"
anon_path = data_dir / "exports" / "anonymized_prices_latest.json"
quarantine_files = list((data_dir / "quarantine").glob("*.json"))

# Load datasets
with open(raw_path, "r", encoding="utf-8") as f:
    raw_records = json.load(f)

with open(cleaned_path, "r", encoding="utf-8") as f:
    cleaned_records = json.load(f)

with open(metrics_path, "r", encoding="utf-8") as f:
    quality_metrics = json.load(f)

with open(board_path, "r", encoding="utf-8") as f:
    rolling_board = json.load(f)

quarantine_records = []
for qf in quarantine_files:
    with open(qf, "r", encoding="utf-8") as f:
        quarantine_records.extend(json.load(f))

print(f"Loaded {len(raw_records)} raw records, {len(cleaned_records)} cleaned records, {len(quarantine_records)} quarantined entries.")

## 1. High-Level Data Health & Quality KPIs
Summary of the automated pipeline audit scorecard.

In [2]:
print("=" * 60)
print("           PIPELINE QUALITY SCORECARD (0 - 100)           ")
print("=" * 60)
print(f"  * Overall Quality Score:    {quality_metrics.get('overall_quality_score')}/100")
print(f"  * Pipeline Validity Rate:   {quality_metrics.get('validity_rate_pct')}%")
print(f"  * Freshness Score:          {quality_metrics.get('freshness_score')}/100")
print(f"  * Completeness Score:       {quality_metrics.get('completeness_score')}%")
print(f"  * Total Raw Ingested:       {quality_metrics.get('total_raw_ingested')}")
print(f"  * Quarantined (Validation): {quality_metrics.get('quarantined_validation_count')}")
print(f"  * Quarantined (Outliers):   {quality_metrics.get('quarantined_outlier_count')}")
print("=" * 60)

## 2. Category Distribution & Price Quantiles
Prices normalized to INR per kilogram across all e-waste streams.

In [3]:
category_groups = {}
for r in cleaned_records:
    category_groups.setdefault(r["category"], []).append(r["normalized_price_per_kg"])

print(f"{'Category':<26} {'N':<6} {'Min':<8} {'P25':<8} {'Median':<8} {'P75':<8} {'Max':<8}")
print("-" * 74)
for cat, prices in sorted(category_groups.items()):
    arr = np.array(prices)
    p_min = f"Rs.{np.min(arr):.1f}"
    p25 = f"Rs.{np.percentile(arr, 25):.1f}"
    med = f"Rs.{np.median(arr):.1f}"
    p75 = f"Rs.{np.percentile(arr, 75):.1f}"
    p_max = f"Rs.{np.max(arr):.1f}"
    print(f"{cat:<26} {len(prices):<6} {p_min:<8} {p25:<8} {med:<8} {p75:<8} {p_max:<8}")

## 3. Geographic Distribution Across Maharashtra Districts

In [4]:
dist_counts = {}
for r in cleaned_records:
    dist_counts[r["district"]] = dist_counts.get(r["district"], 0) + 1

total = len(cleaned_records)
print(f"{'District':<12} {'Sample Count':<14} {'Share':<10} {'Distribution Bar'}")
print("-" * 60)
for dist, cnt in sorted(dist_counts.items(), key=lambda x: -x[1]):
    pct = (cnt / total) * 100
    bar = "#" * int(round(pct / 100 * 30))
    print(f"{dist:<12} {cnt:<14} {pct:>5.1f}%     {bar}")

## 4. Quarantine Root-Cause Breakdown
Audit of anomalous, corrupt, or out-of-bounds observations rejected by the pipeline.

In [5]:
reasons_map = {}
for q in quarantine_records:
    for r in q.get("rejection_reasons", []):
        key = r.split(":")[0] if ":" in r else r[:45]
        reasons_map[key] = reasons_map.get(key, 0) + 1

print("Quarantine Violation Counts:")
for reason, cnt in sorted(reasons_map.items(), key=lambda x: -x[1]):
    print(f"  [{cnt:>2}x] {reason}")

## 5. Privacy & Anonymization Audit
Verification of salted collector ID hashing and ~500m GPS spatial coarsening.

In [6]:
with open(anon_path, "r", encoding="utf-8") as f:
    anon_records = json.load(f)

sample_anon = anon_records[0] if anon_records else {}
print("Sample Anonymized Record for Public / ML Sharing:")
for k, v in sample_anon.items():
    print(f"  {k:<26}: {v}")

# Check for PII presence
pii_keys = ["phone", "name", "address", "aadhaar"]
has_pii = any(k in sample_anon for k in pii_keys)
print(f"\nPII Check: {'PASSED (Zero PII)' if not has_pii else 'FAILED'}")
print(f"Grid Resolution: ~500m (lat={sample_anon.get('coarsened_latitude')}, lng={sample_anon.get('coarsened_longitude')})")